<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Lab5.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 5 — Complete VQE for the H₂ Ground-State Energy
**Quantum Optimization and Simulation — VQE Laboratory Series**

The centerpiece lab: state preparation, Pauli measurements, optimization, and exact comparison.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. In this notebook every bitstring we *discuss* is written in the logical order `q0 q1` (left to right), as in Lab 4. The helper `q0_first(...)` converts Qiskit's labels to this order.

## Learning objectives
- Build a complete reduced two-qubit H₂ VQE.
- Estimate each Pauli contribution using finite shots.
- Minimize the total energy with COBYLA.
- Compare VQE with exact "**diagonalization**".
- *(Optional)* Regenerate the same Hamiltonian using PySCF + Qiskit Nature.

> **NOTE: Diagonalization.**
>- In QAOA, the cost Hamiltonian is already diagonal: each diagonal entry is simply the cost of one bitstring.
>- In VQE, the Hamiltonian matrix is not diagonal, because of its off-diagonal terms (such as $X_0X_1$).
>- Diagonalization means finding a change of basis that makes those off-diagonal terms vanish, leaving the energies on the diagonal. For H₂ this takes a single rotation, but for larger molecules the matrix grows and exact diagonalization quickly becomes impossible, which is why we use VQE.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "pylatexenc" "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

SEED = 123
SHOTS = 4096
HARTREE_TO_EV = 27.2114
HARTREE_TO_KCAL = 627.509
CHEM_ACC = 1.6e-3            # chemical accuracy: 1.6 mHa ≈ 1 kcal/mol

# Helper Functions
def run_counts(qc, shots=SHOTS, noise_model=None, seed=SEED):
    backend = AerSimulator(noise_model=noise_model)
    tqc = transpile(qc, backend, optimization_level=1)
    result = backend.run(tqc, shots=shots, seed_simulator=seed).result()
    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

## The reduced H₂ Hamiltonian
At bond length $R = 0.735$ Å (STO-3G basis), the electronic Hamiltonian of H₂ reduces to two qubits:

$$
H_{\text{elec}} = c_0\, I + c_1\, Z_0 + c_2\, Z_1 + c_3\, Z_0 Z_1 + c_4\,(X_0 X_1 + Y_0 Y_1) .
$$

- $c_0 \dots c_4$ come from the molecular integrals (we regenerate them ourselves in the optional section).
- The qubits follow the Module 3 & 4 convention: $|01\rangle$ (q0-first) is the **bonding** configuration, i.e. the Hartree–Fock reference, and $|10\rangle$ is the **antibonding** configuration.
- $X_0X_1 + Y_0Y_1$ is the **hopping** term: it swaps $|01\rangle \leftrightarrow |10\rangle$ (each of the two terms contributes $c_4$), and on $|00\rangle$ and $|11\rangle$ the two terms cancel. So it mixes the bonding and antibonding configurations without ever leaving the physical subspace.
- The **total** energy also includes the repulsion between the two protons, a constant $E_{\text{nuc}}$ that does not depend on the electrons:

$$E_{\text{total}} = \langle H_{\text{elec}} \rangle + E_{\text{nuc}} .$$

In [ ]:
from scipy.optimize import minimize

# Supplied reduced H2 Hamiltonian coefficients (electronic part, R = 0.735 Å, STO-3G).
c0 = -1.0523732458   # I
c1 = -0.3979374248   # Z0
c2 = +0.3979374248   # Z1
c3 = -0.0112801043   # Z0 Z1
c4 = +0.0904655999   # X0 X1 and Y0 Y1 (hopping)
E_NUC = 0.7199689944 # proton-proton repulsion (Hartree)
E_EXACT = -1.137306

print(f"Exact ground state: electronic electronic -1.857275 Ha + nuclear 0.719969 Ha = TOTAL -1.137306 Ha")

In [ ]:
def h2_ansatz(theta):
    qc = QuantumCircuit(2)

    # Prepare |01> (q0-first): the Hartree-Fock (bonding) reference
    qc.x(1)

    # ------------------------------------------------
    # RYX(theta) = exp[-i theta/2 (Y0 X1)]
    # ------------------------------------------------
    qc.sdg(0)
    qc.h(0)
    qc.h(1)

    # ZZ rotation
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.s(0)
    qc.h(1)

    # ------------------------------------------------
    # RXY(-theta) = exp[+i theta/2 (X0 Y1)]
    # ------------------------------------------------

    qc.h(0)
    qc.sdg(1)
    qc.h(1)

    # ZZ rotation with angle -theta
    qc.cx(0, 1)
    qc.rz(-theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.h(1)
    qc.s(1)
    return qc

h2_ansatz(0.2).draw("mpl")

In [ ]:
def parity_expectation(counts, qubits):
    total = sum(counts.values())
    expval = 0.0

    for bits, count in counts.items():
        bits = bits.replace(" ", "")
        value = 1
        for q in qubits:
            bit = int(bits[-1-q])
            value *= (1 if bit == 0 else -1)
        expval += value * count / total

    return expval

def measure_group(theta, basis, seed=SEED):
    qc = h2_ansatz(theta)

    if basis == "X":
        qc.h([0,1])
    elif basis == "Y":
        qc.sdg([0,1])
        qc.h([0,1])

    qc.measure_all()
    return run_counts(qc, shots=SHOTS, seed=seed)

## Part A — Energy function
Each Pauli term is estimated from measurement counts, using three circuits:
- **Z basis:** $Z_0$, $Z_1$ and $Z_0Z_1$ all come from one circuit measured as is.
- **X basis:** $X_0X_1$ needs $H$ gates before measurement.
- **Y basis:** $Y_0Y_1$ needs $S^\dagger$ then $H$ before measurement.

In [ ]:
evaluation_history = []

def pauli_terms_from_shots(theta, seed=SEED):
    z_counts = measure_group(theta, "Z", seed)
    x_counts = measure_group(theta, "X", seed)
    y_counts = measure_group(theta, "Y", seed)
    return {
        "Z0":   parity_expectation(z_counts, [0]),
        "Z1":   parity_expectation(z_counts, [1]),
        "Z0Z1": parity_expectation(z_counts, [0, 1]),
        "X0X1": parity_expectation(x_counts, [0, 1]),
        "Y0Y1": parity_expectation(y_counts, [0, 1]),
    }

def energy_from_shots(theta_array, seed=SEED):
    theta = float(np.atleast_1d(theta_array)[0])
    t = pauli_terms_from_shots(theta, seed)
    energy = c0 + c1*t["Z0"] + c2*t["Z1"] + c3*t["Z0Z1"] + c4*(t["X0X1"] + t["Y0Y1"]) + E_NUC
    evaluation_history.append((theta, energy))
    return energy

## Part B — Hartree–Fock Point
Let's look inside one energy evaluation at the Hartree–Fock point, $\theta = 0$, where both electrons are in the bonding orbital.

In [ ]:
theta_hf = 0.0
terms = pauli_terms_from_shots(theta_hf)
coef = {"Z0": c1, "Z1": c2, "Z0Z1": c3, "X0X1": c4, "Y0Y1": c4}
cname = {"Z0": "c1", "Z1": "c2", "Z0Z1": "c3", "X0X1": "c4", "Y0Y1": "c4"}
contrib = {"I (c0)": c0, **{f"{k} ({cname[k]})": coef[k] * terms[k] for k in coef}, "nuclear": E_NUC}

print(f"{'term':10}{'coefficient':>13}{'<P> (shots)':>14}{'contribution':>15}")
print(f"{'I':10}{c0:13.5f}{1.0:14.4f}{c0:15.5f}")
for k in coef:
    print(f"{k:10}{coef[k]:13.5f}{terms[k]:14.4f}{coef[k]*terms[k]:15.5f}")
print(f"{'nuclear':10}{'':13}{'':14}{E_NUC:15.5f}")
E_HF = energy_from_shots([theta_hf])
print(f"\nHartree-Fock energy (shots): {E_HF:.5f} Ha") #     exact statevector: {exact_energy(theta_hf):.5f} Ha")

plt.figure(figsize=(8, 3.5))
bars = plt.bar(list(contrib.keys()), list(contrib.values()),
               color=["gray", "tab:blue", "tab:blue", "tab:orange", "tab:green", "tab:green", "tab:red"])
plt.axhline(0, color="black", lw=0.8)
plt.ylabel("Contribution (Hartree)"); plt.title("Where the Hartree–Fock energy comes from (θ = 0)")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

At $\theta = 0$ the state is exactly $|01\rangle$, so $\langle Z_0\rangle = +1$, $\langle Z_1\rangle = -1$, $\langle Z_0Z_1\rangle = -1$, and $\langle X_0X_1\rangle \approx \langle Y_0Y_1\rangle \approx 0$: the hopping term, which mixes in the antibonding configuration, does nothing yet. VQE will lower the energy by turning $\theta$ so that this term starts to contribute.

## Part C — Optimize with COBYLA

In [ ]:
evaluation_history.clear()

result = minimize(
    energy_from_shots,
    x0=np.array([0.0]),
    method="COBYLA",
    options={"maxiter": 45, "rhobeg": 0.4}
)

theta_opt = result.x[0]
E_VQE = result.fun                       # what COBYLA reports (a noisy estimate)
print(f"COBYLA finished after {result.nfev} energy evaluations ({result.message})")
print(f"{'':28}{'Hartree':>12}{'eV':>12}")
print(f"{'Hartree-Fock (theta = 0)':28}{E_HF:12.6f}{E_HF*HARTREE_TO_EV:12.4f}")
print(f"{'VQE (shots, reported)':28}{E_VQE:12.6f}{E_VQE*HARTREE_TO_EV:12.4f}")
print(f"{'Exact (diagonalization)':28}{E_EXACT:12.6f}{E_EXACT*HARTREE_TO_EV:12.4f}")
print(f"Optimal theta = {theta_opt:.4f} rad")

hist = np.array(evaluation_history)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(hist[:,1], marker=".", label="VQE energy (shots)")
axes[0].axhline(E_EXACT, color="red", ls="--", label="exact ground state")
axes[0].axhline(E_HF, color="gray", ls=":", label="Hartree-Fock")
axes[0].set_xlabel("Energy evaluation"); axes[0].set_ylabel("Energy (Hartree)")
axes[0].set_title("COBYLA VQE convergence"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(hist[:,0], marker=".", color="tab:purple")
axes[1].axhline(theta_opt, color="black", ls="--", label=f"final theta = {theta_opt:.3f}")
axes[1].set_xlabel("Energy evaluation"); axes[1].set_ylabel("theta (rad)")
axes[1].set_title("Path of the control knob"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Part D — Report the optimized mix

In [ ]:
qc_opt = h2_ansatz(theta_opt)
qc_opt.measure_all()
counts = run_counts(qc_opt, shots=8192)
total = sum(counts.values())
probs = {q0_first(k): v / total for k, v in counts.items()}

print("Raw Qiskit counts (q1 q0):", counts)
print(f"P(01) bonding     = {probs.get('01', 0):.4f}   theory cos^2(theta) = {np.cos(theta_opt)**2:.4f}")
print(f"P(10) antibonding = {probs.get('10', 0):.4f}   theory sin^2(theta) = {np.sin(theta_opt)**2:.4f}")

labels = ["00", "01", "10", "11"]
measured = [probs.get(l, 0) for l in labels]
theory = [0, np.cos(theta_opt)**2, np.sin(theta_opt)**2, 0]
x = np.arange(4)
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, measured, 0.4, label="measured (8192 shots)")
ax.bar(x + 0.2, theory, 0.4, label="theory")
ax.set_xticks(x, ["00\n(unphysical)", "01\n(bonding)", "10\n(antibonding)", "11\n(unphysical)"])
for xi, m in zip(x, measured):
    ax.text(xi - 0.2, m + 0.01, f"{m:.3f}", ha="center", fontsize=8)
ax.set_ylabel("Probability"); ax.set_title("Optimized H2 configuration mix (q0-first labels)"); ax.legend()
plt.show()

The optimized ground state is about **98% bonding and 2% antibonding**. That small antibonding admixture is the *electron correlation* that Hartree–Fock misses, and it is worth about 20 mHa of energy.


## Reflection
Why can a shot-based VQE energy occasionally appear slightly below the exact ground-state energy even though the variational principle says it should not?



A finite-shot estimate contains statistical error, so the *estimate* can fluctuate slightly below the exact value even though the underlying state does not violate the variational principle.

## Optional — Compute the H₂ ground-state energy with PySCF and Qiskit Nature
So far the Hamiltonian coefficients were *supplied*. Here we compute the H₂ ground-state energy from scratch in three steps:
1. **PySCF** describes the molecule and computes its orbitals and energy integrals.
2. **Qiskit Nature** turns that into a qubit Hamiltonian, one qubit per spin orbital (the four "lanes" of Lab 4).
3. **VQE** finds the lowest energy, using ready-made building blocks from Qiskit Nature.

In [ ]:
# This optional section requires PySCF, which may take longer to install.
%pip -q install pyscf

### Step 1 — Describe the molecule (PySCF)

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver

driver = PySCFDriver(
    atom="H 0 0 0; H 0 0 0.735",   # two H atoms, 0.735 Å apart
    basis="sto3g"                   # minimal basis: one 1s orbital per atom
)
problem = driver.run()                                   # PySCF computes orbitals and integrals
second_q_hamiltonian = problem.hamiltonian.second_q_op()  # electron Hamiltonian (a†, a operators)
E_NUC_PYSCF = problem.nuclear_repulsion_energy           # proton-proton repulsion (constant)

print("spatial orbitals :", problem.num_spatial_orbitals)
print("electrons (up, down):", problem.num_particles)
print(f"nuclear repulsion: {E_NUC_PYSCF:.6f} Ha")

### Step 2 — Map to qubits (Qiskit Nature, Jordan–Wigner)

In [ ]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

qubit_hamiltonian_4q = JordanWignerMapper().map(second_q_hamiltonian)

print(f"{qubit_hamiltonian_4q.num_qubits} qubits, {len(qubit_hamiltonian_4q)} Pauli terms (Qiskit labels: q3 q2 q1 q0)")
for label, coeff in qubit_hamiltonian_4q.to_list():
    print(f"  {coeff.real:+.6f}  {label}")

### Step 3 — Solve it with VQE (Qiskit Nature + Qiskit Algorithms)
This is the same VQE recipe as Parts A–C, with each ingredient now supplied by a library:

| Ingredient | Parts A–C (by hand) | Here (libraries) |
|---|---|---|
| Hamiltonian | supplied $c_0 \dots c_4$, 2 qubits | from PySCF (Steps 1–2), 4 qubits |
| Starting state | `qc.x(1)` → Hartree–Fock, $\theta = 0$ | `HartreeFock(...)` → $|1010\rangle$, all parameters $= 0$ |
| Ansatz | `h2_ansatz(theta)`, 1 parameter | `UCCSD(...)`, built automatically |
| Energy evaluation | measure Z, X, Y circuits on `AerSimulator`, 4096 shots each | `BackendEstimatorV2` on `AerSimulator`, 4096 shots per circuit (it groups the Pauli terms into circuits for us) |
| Optimizer | COBYLA (`maxiter=45`, `rhobeg=0.4`) | COBYLA (`maxiter=45`, `rhobeg=0.4`) |
| Put together | our own loop | `VQE` inside `GroundStateEigensolver` |

In [ ]:
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD

mapper = JordanWignerMapper()
hf_state = HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
ansatz = UCCSD(problem.num_spatial_orbitals, problem.num_particles, mapper, initial_state=hf_state)

print(f"UCCSD ansatz: {ansatz.num_qubits} qubits, {ansatz.num_parameters} parameters, one per excitation:")
for k, (frm, to) in enumerate(ansatz.excitation_list):
    kind = "single" if len(frm) == 1 else "double"
    print(f"  theta[{k}]: {kind} excitation, lane(s) {frm} -> {to}")

These are exactly the excitations from the lecture: lane 0→1 ($T_C$, spin up), lane 2→3 ($T_B$, spin down), and both together ($T_D$, the double excitation).

In [ ]:
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA
from qiskit.primitives import BackendEstimatorV2
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
import warnings
from scipy.sparse import SparseEfficiencyWarning
warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)   # harmless scipy message

# Same simulator, shots, and seed as run_counts() in Parts A-C
backend = AerSimulator()
estimator = BackendEstimatorV2(
    backend=backend,
    options={"default_precision": 1 / np.sqrt(SHOTS),   # precision 1/sqrt(4096) -> 4096 shots per circuit
             "seed_simulator": SEED},
)
ansatz_t = transpile(ansatz, backend, optimization_level=1)   # as run_counts() does

vqe_history = []
vqe = VQE(
    estimator=estimator,                       # measures <H> from shots for a given set of parameters
    ansatz=ansatz_t,
    optimizer=COBYLA(maxiter=45, rhobeg=0.4),  # same optimizer settings as Part C
    callback=lambda count, params, energy, meta: vqe_history.append(energy),
)
vqe.initial_point = np.zeros(ansatz.num_parameters)   # start from Hartree-Fock, like x0 = 0 in Part C

solver = GroundStateEigensolver(mapper, vqe)  # maps the problem to qubits, runs VQE, adds nuclear repulsion
result_nature = solver.solve(problem)

E_vqe_nature = result_nature.total_energies[0]
print(f"electronic energy : {result_nature.groundenergy:.6f} Ha")
print(f"nuclear repulsion : {E_NUC_PYSCF:.6f} Ha")
print(f"TOTAL energy (VQE): {E_vqe_nature:.6f} Ha  ({E_vqe_nature * HARTREE_TO_EV:.4f} eV)")
print("\noptimal parameters:", np.round(result_nature.raw_result.optimal_point, 4) + 0.0)

# ---- Plot on the same energy scale as the Part C convergence graph ----
part_c = hist[:, 1]                                    # VQE energies from Part C
lo = min(part_c.min(), E_EXACT, E_HF)
hi = max(part_c.max(), E_EXACT, E_HF)
pad = 0.05 * (hi - lo)                                 # matplotlib's default 5% margin, as in Part C

plt.figure(figsize=(6.5, 4.5))                         # same size as each Part C panel
plt.plot(part_c, marker=".", color="tab:blue", alpha=0.5, label="Part C: VQE energy (shots)")
plt.plot(np.array(vqe_history) + E_NUC_PYSCF, marker=".", color="tab:orange",
         label="PySCF + Qiskit Nature (UCCSD)")
plt.axhline(E_EXACT, color="red", ls="--", label="exact ground state")
plt.axhline(E_HF, color="gray", ls=":", label="Hartree-Fock")
plt.ylim(lo - pad, hi + pad)
plt.xlabel("Energy evaluation"); plt.ylabel("Energy (Hartree)")
plt.title("COBYLA VQE convergence: Part C vs PySCF + Qiskit Nature")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()

**What to notice**
- The total energy lands within a few mHa of **−1.137306 Ha**, the H₂ ground-state energy we found at the start of the lab. As in Part C, the small difference is shot noise.
- The two single-excitation parameters end **close to 0**: for H₂, single excitations do not lower the energy (the symmetry argument from the lecture and Lab 4). They are not exactly 0 only because of shot noise.
- The double-excitation parameter ends near **−0.12**, close to the angle COBYLA found for our hand-built circuit in Part C. Both VQEs prepare the same state: mostly bonding with a small antibonding admixture.
- The convergence plot looks like Part C's: COBYLA explores, then settles near the minimum.